<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_27_pipeline_weblog_qa_reprocessing.ipynb">9.27 Pipeline QA、Weblog 与再处理决策</a></li>
        <li>下一节： <a href="9_29_catalog_detection_product_validation.ipynb">9.29 源表与检测产品验证</a></li>
    </ul>
</div>


## 9.28 最小可复现处理项目：从配置文件到验证闭环

第 9.24 节讨论了软件生态和 provenance，第 9.27 节讨论了 pipeline weblog 与再处理决策。两者之间还缺少一个非常实际的环节：当归档数据、pipeline 产品、手动重成像、源表生成和科学测量都进入同一个研究项目时，项目本身应如何组织，才能让结果在半年后仍然可以被审查、重跑和解释。

射电干涉数据处理的复杂性不只来自算法。一个连续谱或谱线项目常常同时包含大型 Measurement Set、校准表、flag 版本、成像参数、mask、残差图、源表、区域文件、测量脚本、可视化截图、手工判断、pipeline weblog 和最终论文图。若这些对象混在一个工作目录中，项目会很快退化成“最后一张图看起来不错，但无法说明它怎样产生”。可复现项目模板的意义，就是把数据身份、软件身份、参数身份、处理依赖和验证证据固定下来，使科学测量不再依赖个人记忆。

这一节的目标不是引入大型工作流系统，也不是把所有项目都变成工程化流水线。真正需要的是一个足够小、足够清楚、能够随项目增长而扩展的结构。它应能服务单人小课题，也能服务小组内的归档数据再处理；它应能容纳 CASA、WSClean、DP3、DDFacet、PyBDSF、SoFiA、CARTA 和 Python 测量脚本，同时不把软件名误当成项目结构。


### 9.28.1 可复现不等于保存一个 notebook

Notebook 很适合解释思路、展示诊断图和记录科学测量，但它不应该成为整个项目的唯一运行入口。原因很简单：notebook 的执行顺序可能不同于单元格显示顺序；交互检查中产生的中间变量可能没有被完整记录；本机路径、临时文件名和隐藏状态很容易进入计算；大型数据对象也通常不会真正保存在 notebook 内部。对于射电干涉项目，这些问题会被放大，因为一个小参数改动可能要求重新生成多个下游产品。

更稳健的表述是把处理项目看成一个从输入身份到验证证据的映射：

$$
(P,Q)=F(D,E,\boldsymbol{\theta},G),
$$

其中 $D$ 表示数据身份，$E$ 表示软件和运行环境，$\boldsymbol{\theta}$ 表示参数集合，$G$ 表示处理步骤之间的依赖图，$P$ 是图像、cube、源表和测量表等产品，$Q$ 是噪声、残差、校准解、通量恢复和单位检查等验证证据。可复现性要求的不只是重新得到一个文件名相同的结果，而是能够说明同一个 $D$、同一个 $E$、同一个 $\boldsymbol{\theta}$ 和同一个 $G$ 如何产生同一组可解释的 $P$ 与 $Q$。

这个公式也说明了为什么“把命令复制到 notebook 里”并不充分。命令只是 $F$ 的一部分，参数来源、输入数据版本、运行环境和验证门同样重要。若只有最终图像而没有 $D$，无法判断它来自哪个 execution block 或哪个 flag 版本；若只有脚本而没有 $E$，无法判断软件默认值是否改变；若只有参数而没有 $G$，无法知道修改一个阈值后哪些产品需要重算；若只有产品而没有 $Q$，无法判断图像是否达到科学目标。


![最小可复现项目结构](figures/practical_minipipeline_project_tree.png)

图中把一个最小射电干涉项目分为数据、配置、脚本、日志、产品、笔记和验证七类对象。大数据不必进入版本库，但必须用稳定标识、校验和和下载记录引用；配置、脚本、日志、小型质控图和说明文本应尽量进入版本控制。这样做的核心不是追求形式整齐，而是让项目身份能够脱离个人机器存在。


### 9.28.2 项目目录：把“可运行对象”和“解释对象”分开

一个小型连续谱再处理项目可以从如下结构开始：

```text
project_name/
  data/
    raw_manifest.yml
    checksums.txt
    README_data.md
  configs/
    science_goal.yml
    calibration.yml
    imaging.yml
    catalog.yml
    validation.yml
  scripts/
    fetch_archive.sh
    run_flagging.sh
    run_calibration.sh
    run_imaging.sh
    run_catalog.sh
    make_validation_plots.py
  logs/
    pipeline_weblog_link.txt
    command_history.txt
    manual_decisions.md
  products/
    images/
    cubes/
    catalogs/
    masks/
  notebooks/
    inspect_weblog.ipynb
    measure_fluxes.ipynb
  validation/
    noise_residuals.png
    flux_scale_check.csv
    qa_report.md
```

这个目录树的关键在于职责分离。`data/` 不存放难以管理的大型 Measurement Set，而是存放数据清单、归档链接、下载日期、execution block、field、spectral window、scan、校准意图和校验和。真正的大型数据可以在共享存储、对象存储或本地数据盘中；项目只需要可靠地指向它。`configs/` 存放可运行参数，是处理链的参数来源。`scripts/` 只做薄包装，把配置文件转化为 CASA、WSClean、DP3、PyBDSF、SoFiA 或 Python 命令。`logs/` 保存机器日志和人工判断。`products/` 保存可再生成产品。`notebooks/` 用于解释、检查和科学测量。`validation/` 保存用来判定结果是否可信的证据。

这种结构有一个重要后果：笔记不再负责携带所有状态。笔记可以解释为什么选用某个 Briggs robust 值，展示残差图，讨论是否需要重新 mask；但成像参数本身应出现在 `configs/imaging.yml` 中。科学测量笔记可以调用源表和图像，给出通量、亮温或谱指数；但源搜索阈值、岛屿合并条件和局部 RMS 设置应出现在 `configs/catalog.yml` 中。这样半年后重新运行项目时，参数不会散落在多个单元格、多个临时命令和多个截图里。


### 9.28.3 参数从科学目标进入配置文件

可复现项目的参数不应从软件默认值开始，而应从科学目标开始。若目标是分辨一对角距约 $5''$ 的致密源，成像参数必须保证合成波束、像元大小和权重选择能够支持这一目标；若目标是测量弥散发射总通量，最大可恢复尺度、短基线权重、taper、清洁 mask 和主波束边缘必须成为一等参数；若目标是寻找弱谱线，通道宽度、连续谱扣除区间、速度约定和 line-free 通道定义必须显式记录。

几个常用尺度把科学目标和配置文件连接起来。角分辨率近似由

$$
\theta_{\rm res}\sim \frac{\lambda}{b_{\rm max}}
$$

控制；最大可恢复尺度常用量级

$$
\theta_{\rm MRS}\sim 0.6\frac{\lambda}{b_{\rm min}}
$$

估计；像元大小通常需要比合成波束小数倍，以便稳定描述点扩散函数和拟合源尺寸。噪声目标可由辐射计方程估计：

$$
\sigma_{\rm th}\simeq \frac{\mathrm{SEFD}}{\eta\sqrt{n_{\rm pol}N(N-1)\Delta\nu t}},
$$

其中 $N$ 是天线数，$\Delta\nu$ 是有效带宽，$t$ 是有效积分时间，$\eta$ 汇总相关效率、flag 损失和加权损失。真实图像噪声若显著高于这个估计，可能来自残余校准误差、混淆噪声、方向相关效应、清洁不足或数据权重问题。

一个配置片段可以这样表达科学目标与软件参数之间的关系：

```yaml
science:
  field: target_A
  goal: diffuse_emission_and_compact_catalog
  required_resolution: 8 arcsec
  largest_scale_of_interest: 3 arcmin
  expected_rms: 15 uJy_per_beam

imaging:
  cell: 1.5 arcsec
  imsize: [4096, 4096]
  weighting: briggs
  robust: 0.5
  taper: 8 arcsec
  nterms: 2
  primary_beam_limit: 0.3

catalog:
  finder: pybdsf
  island_threshold: 3.0
  pixel_threshold: 5.0
  rms_box: [80, 20]

validation:
  rms_tolerance_factor: 1.3
  residual_peak_fraction: 0.01
  flux_scale_tolerance: 0.10
```

这个例子中的软件名不是重点。重点是参数有来源：分辨率目标进入 `cell`、`taper` 和权重；弥散结构目标进入最大尺度判断和短基线权重；源表目标进入 PyBDSF 阈值和局部 RMS 估计；验证目标进入噪声、残差和通量尺度容许范围。参数一旦进入配置文件，就可以被版本控制、比较、复用和审查。


![参数流向配置文件](figures/practical_minipipeline_config_parameter_flow.png)

图中强调“参数唯一来源”。科学需求先进入配置文件，再由薄脚本生成具体软件命令。CASA、WSClean、DP3、PyBDSF、SoFiA 或测量 notebook 可以消费这些参数，但不应各自保存一套互相矛盾的隐式取值。


### 9.28.4 依赖图：重跑范围必须可解释

射电处理中的许多错误来自重跑范围不清。若只改了源搜索阈值，通常不需要重新校准和成像；若改了 flag 版本，校准表、改正后数据、图像、源表和测量都可能失效；若改了成像权重或 taper，源表和图像测量必须更新，校准日志却可以保留；若改了 flux calibrator 模型，则通量尺度相关的所有产品都需要重新审查。

这种关系适合用有向无环图表示。设处理节点集合为 $V$，边 $u\rightarrow v$ 表示产品 $v$ 依赖产品或参数 $u$。当参数 $\theta_i$ 发生改变时，需要重跑的节点集合可以写成

$$
R(\theta_i)=\{v\in V:\;\theta_i\rightarrow\cdots\rightarrow v\},
$$

也就是从该参数出发能够到达的所有下游节点。这个集合不必由复杂软件自动计算；在小项目中，一个清楚的流程图、一个轻量级构建脚本或一份 `Makefile` 风格规则就足以避免很多混乱。

```text
raw_ms       -> flagged_ms
flagged_ms  -> cal_tables -> corrected_ms
corrected_ms -> image_cube -> catalog_mask -> science_table -> report
flagged_ms  -> flag_qa
cal_tables  -> cal_qa
image_cube  -> image_qa
science_table -> measurement_qa
```

依赖图还改变了日志的地位。日志不是事后附加说明，而是每个节点的伴随产品。flagging 需要记录自动阈值、人工 flag 原因和保留的数据比例；校准需要记录参考天线、解区间、失败天线、解振幅和相位稳定性；成像需要记录权重、taper、mask、阈值、迭代停止条件和残差；源表需要记录阈值、局部 RMS、岛屿合并、拟合失败和可靠性判断。若每一步都有日志，失败模式就能被定位到处理链中的具体位置。


![小型流程依赖图](figures/practical_minipipeline_workflow_dag.png)

图中把原始测量集、标记、校准、成像、源表、科学量表和报告包串成依赖链，并在关键节点旁放置日志和质控图。参数改变并不意味着盲目完整重跑；依赖图说明哪些下游产品必须更新，哪些上游证据可以保留。


### 9.28.5 数据身份、软件身份与处理身份

可复现性经常被误解为“软件版本一致”。软件版本当然重要，但它只是三类身份之一。第一类是数据身份：项目编号、观测日期、execution block、field、scan、spectral window、intent、flag 版本、原始文件校验和、下载日期和 pipeline 产品编号。归档数据尤其需要记录这些信息，因为同一个项目可能有多个 execution block、多个 pipeline 版本或重新发布的校准产品。

第二类是软件身份。CASA、WSClean、DP3、DDFacet、PyBDSF、SoFiA、Astropy、numpy、scipy、radio-beam、spectral-cube 和 regions 的版本都可能影响结果。某些差异来自算法改进，某些来自默认参数改变，某些来自底层库、编译选项或容器镜像。对教学项目而言，不一定需要从第一天就构建完整容器，但至少应保存环境文件、关键命令输出、软件版本和操作系统信息。对需要长期复现的项目，容器、模块环境或锁定依赖文件会更稳健。

第三类是处理身份。它包括配置文件哈希、代码提交号、命令日志、人工记录、mask 版本、region 文件、失败尝试和最终采用理由。处理身份常常最容易丢失，因为它位于自动 pipeline 和人工判断之间。比如一个源表可能来自 PyBDSF，但局部 RMS box、island threshold、人工排除的边缘伪源和最终合并规则若没有记录，源表就不能被严格复现。一个图像可能来自 WSClean 或 CASA tclean，但 mask 如何产生、停止阈值如何选择、是否做过主波束校正和是否应用了 taper，都必须成为处理身份的一部分。


![数据软件处理身份](figures/practical_minipipeline_environment_checksum.png)

图中把可复现运行记录分解为数据身份、软件身份和处理身份。三者缺一不可。只有数据而没有软件环境，重跑可能因默认值改变而偏离；只有软件而没有数据校验和，无法确认输入是否相同；只有产品而没有处理身份，无法解释人工选择如何影响结果。


### 9.28.6 验证门：质控不应是最后一页

可复现项目的另一个关键，是把验证目标放在处理开始之前。质控若只在最后写成一页总结，往往已经太晚；到那时，残差结构、通量尺度、单位错误或源表偏差可能已经传播到科学结论中。更稳健的做法是为每个关键步骤定义 validation gate，使失败能够在合适位置停止。

对连续谱成像而言，常见验证门包括：flag 后保留数据比例是否合理，坏天线或坏频段是否集中；bandpass 和 gain 解是否随时间、频率和天线平滑变化；成像后远离亮源的 RMS 是否接近热噪声估计；残差峰值是否由真实未建模结构、清洁不足或校准误差造成；恢复波束是否符合预期，单位是否保持为 Jy/beam；主波束校正后噪声是否随半径增长；注入源或已知源的通量是否在误差范围内恢复；源表的假阳性是否集中在边缘、旁瓣或残差结构附近。

对谱线项目，验证门还要包括速度轴、频率参考系、连续谱扣除区间、line-free 通道、moment mask、噪声相关性和柱密度转换。对偏振项目，验证门还要包括泄漏、交叉手相位、Ricean bias、RMSF、Faraday 旁瓣和电离层 RM。对低频与高频项目，验证门则分别强调电离层方向相关残差、RFI 后的频率覆盖、天气、opacity、系统温度和相干损失。

验证门不意味着设置僵硬的通过线。真实数据经常有灰区，更合理的做法是把每个门分为“可接受”“需要解释”“停止并回到上游”三类。比如图像 RMS 高于预期 20% 可能只需要记录 weighting 和 flag 损失；高于预期数倍且伴随亮源旁瓣，则应回到校准和成像模型；源表中大量边缘源没有主波束校正误差标记，则应停止源表发布并回到测量步骤。


![验证目标与质控闭环](figures/practical_minipipeline_validation_targets.png)

图中左侧给出一个验证门矩阵：不同检查项应落在不同处理阶段，而不是全部堆到最后。右侧强调质控是闭环；每次运行步骤后生成证据，再与科学需求比较，随后接受产品或修改配置并重跑受影响节点。


### 9.28.7 教学案例：一个小型连续谱再处理项目

设想一个归档连续谱数据集，pipeline 已经给出校准产品和 weblog，原项目目标是探测场中心致密源。新的科学目标是同时测量一片低表面亮度弥散发射，并生成场内致密源表。这个目标改变了项目的关注点：pipeline 图像的中心 RMS 仍然重要，但短基线权重、taper、主波束边缘、弥散结构通量恢复、源表阈值和局部 RMS 估计变得同样重要。

最小项目首先建立 `data/raw_manifest.yml`，记录归档项目编号、execution block、field、spectral window、下载日期、pipeline 产品链接、weblog 位置和输入文件校验和。随后建立 `configs/science_goal.yml`，明确目标波束、最大尺度、目标 RMS、主波束使用范围、是否需要源表和最终测量量。`configs/imaging.yml` 保存成像参数：权重、robust、taper、cell、imsize、多尺度清洁尺度、阈值、mask 生成方式、主波束限制和是否输出残差。`configs/catalog.yml` 保存 PyBDSF 的阈值、RMS box、岛屿合并、边缘排除和输出列。`configs/validation.yml` 保存期望噪声、残差峰值比例、通量恢复容许范围、源表可靠性检查和单位检查。

第一次运行保留 pipeline 校准表，只重新成像。若成像质控显示中心 RMS 接近期望，但弥散结构周围有明显负碗，项目应回到配置文件检查最大可恢复尺度、短基线权重、mask 和是否需要短间距补充；若残差主要集中在强致密源旁，可能需要更好的源模型或保守自校准；若主波束边缘出现大量源表候选，则 PyBDSF 阈值、局部 RMS box 和主波束限制需要调整。每一次调整都应形成新的配置提交，而不是覆盖旧参数。

这个案例的教学价值在于把“做一张图”转化为可审查闭环。弥散发射通量不是由最后一次 aperture measurement 单独决定，而是由数据身份、pipeline weblog、成像配置、mask、短基线响应、残差结构、主波束校正、源表排除和误差预算共同决定。源表也不是 PyBDSF 输出文件本身，而是阈值、局部噪声、边缘处理、残差伪源排查和版本记录共同构成的产品。项目模板使这些证据可以被定位、比较和重跑。


### 9.28.8 本节结论

最小可复现处理项目不是额外的行政负担，而是射电干涉测量链的一部分。数据身份回答“输入是什么”，软件身份回答“用什么环境处理”，处理身份回答“怎样处理”，依赖图回答“改动后重跑哪里”，验证门回答“结果是否满足科学目标”。这五个问题一旦被项目结构固定下来，pipeline 产品、手动重成像、PyBDSF 源表、SoFiA 谱线搜索、CARTA 检查和 Python 测量脚本就能进入同一个可审查框架。

第 9.28 节把第 9 章的实践链进一步推进到项目层面。前面的章节说明如何检查、flag、校准、成像、自校准、做源表、处理谱线、偏振、短间距、宽场、VLBI、特殊频段和 pipeline weblog；这一节说明这些步骤如何组织成一个能被保存、解释、重跑和教学复盘的小型项目。后续若继续加厚，可以在此基础上加入真实公开数据的目录模板、配置文件仓库、容器环境、轻量级构建脚本和失败案例对照。
